# AfriVoices — GÉNÉRATEUR DE SOUMISSION (réutilisable pour tout modèle)

Notebook autonome : produit `submission_{TAG}.csv` (41 733 lignes) pour n'importe quel
modèle w2v-BERT de la lignée (v5-t3, v6-t0, v6-t1, ...) + KenLM au choix.

**Le SEUL bloc à régler : la §2** (`MODEL_NAME`, `LM_DIR`, `ALPHA`, `BETA`, `TAG`).

**Mode d'emploi (session GPU fraîche)** :
1. §1 : installe kenlm + pyctcdecode → **REDÉMARRE le runtime** → relance §1 (vérif).
2. §2 (config) → §3 (rechargement) → §4 (test data) → §5 (inférence, ~30-45 min,
   reprise auto en cas de coupure) → §6 (assemblage + asserts + CSV).

**Robustesse coupures** : l'inférence sauvegarde un fichier partiel sur le Drive tous
les 10 parquets. Après une déconnexion : relancer §1→§4 puis §5 — elle reprend où elle
en était.

## 1. Dépendances (redémarrer le runtime après la 1ère exécution)

In [ ]:
import importlib
need = []
for m in ["kenlm","pyctcdecode"]:
    try: importlib.import_module(m)
    except ImportError: need.append(m)
if need:
    print("installation de", need, "...")
    import subprocess
    subprocess.run(["pip","install","-q","pyctcdecode"], check=False)
    subprocess.run(["pip","install","-q","https://github.com/kpu/kenlm/archive/master.zip"], check=False)
    print("⚠️ REDÉMARRE le runtime (Exécution > Redémarrer la session), puis relance cette cellule.")
else:
    print("✓ kenlm + pyctcdecode disponibles — continue en §2")

## 2. CONFIG — le seul bloc à modifier entre les modèles

In [ ]:
# ============================================================
MODEL_NAME = "baobab-asr-v6-t0"     # <- le modèle à soumettre (dossier sur le Drive)
LM_SUBDIR  = "lm_v2"                # <- "lm_v2" (recommandé) ou "lm"
ALPHA, BETA = 0.5, 1.0              # <- réglage gagnant de la grille
TAG = "v6t0_lmv2"                   # <- suffixe du fichier de soumission
# ============================================================

from google.colab import drive
drive.mount("/content/drive")
import os
BASE  = "/content/drive/MyDrive/afrivoices"
MODEL = f"{BASE}/{MODEL_NAME}"
LM    = f"{BASE}/{LM_SUBDIR}"
PARTIEL = f"{BASE}/submission_{TAG}_partiel.csv"
FINAL   = f"{BASE}/submission_{TAG}.csv"
assert os.path.isdir(MODEL), f"modèle absent: {MODEL}"
assert os.path.isdir(LM), f"LM absent: {LM}"
print(f"modèle: {MODEL_NAME} | LM: {LM_SUBDIR} | alpha={ALPHA} beta={BETA}")
print(f"sortie: {FINAL}")

## 3. Rechargement : modèle + labels + décodeurs + decode_robuste

In [ ]:
import torch, io, base64, os, numpy as np, soundfile as sf, librosa, tempfile
from transformers import Wav2Vec2BertForCTC, Wav2Vec2BertProcessor
from pyctcdecode import build_ctcdecoder
from collections import Counter

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device, "" if device=="cuda" else "⚠️ GPU recommandé (30-45 min vs plusieurs heures)")

model = Wav2Vec2BertForCTC.from_pretrained(MODEL, local_files_only=True).to(device).eval()
processor = Wav2Vec2BertProcessor.from_pretrained(MODEL, local_files_only=True)

raw = [t for t,_ in sorted(processor.tokenizer.get_vocab().items(), key=lambda kv: kv[1])]
blank_tok = processor.tokenizer.pad_token
labels, n = [], 0
for t in raw:
    if t == blank_tok: labels.append("")
    elif t in ("|"," "): labels.append(" ")
    elif t in {"[UNK]","<s>","</s>"}: n += 1; labels.append("\u2047"*n)
    else: labels.append(t)
assert labels.count("")==1 and labels.count(" ")==1

def decode_robuste(b):
    if not b: raise ValueError("vide")
    try: arr, sr = sf.read(io.BytesIO(b), dtype="float32")
    except Exception:
        try: arr, sr = sf.read(io.BytesIO(base64.b64decode(b)), dtype="float32")
        except Exception:
            with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as t:
                try: t.write(base64.b64decode(b))
                except Exception: t.write(b)
                p = t.name
            arr, sr = librosa.load(p, sr=16000, mono=True); os.unlink(p)
            return arr.astype(np.float32)
    if arr.ndim > 1: arr = arr.mean(axis=1)
    if sr != 16000: arr = librosa.resample(arr, orig_sr=sr, target_sr=16000)
    return arr.astype(np.float32)

def unigrams(lang, top=50000):
    c = Counter()
    with open(f"{LM}/{lang}.txt", encoding="utf-8") as f:
        for line in f: c.update(line.split())
    return [w for w,_ in c.most_common(top)]

decoders = {}
for lang in ["swa","kik","luo","kln","mas","som"]:
    decoders[lang] = build_ctcdecoder(labels, kenlm_model_path=f"{LM}/{lang}.bin",
                                      unigrams=unigrams(lang), alpha=ALPHA, beta=BETA)
print("✓ prêt :", MODEL_NAME, "+", LM_SUBDIR, f"(alpha={ALPHA}, beta={BETA})")
print("(warnings pyctcdecode 'length > 1' / 'unigrams don\'t agree' = bénins)")

## 4. Données de test (Drive si présent, sinon téléchargement HF en local)

In [ ]:
import glob
TEST_CANDIDATES = [f"{BASE}/test", "/content/test_data"]
parquets = []
for td in TEST_CANDIDATES:
    parquets = sorted(glob.glob(f"{td}/**/*.parquet", recursive=True))
    if parquets:
        TEST_DIR = td; break
if not parquets:
    print("test absent localement -> téléchargement HF (quelques Go)...")
    from huggingface_hub import snapshot_download, login
    # login("hf_...")   # décommente si le repo test est gated pour ton compte
    TEST_DIR = "/content/test_data"
    snapshot_download("digitalumuganda/anv-test-data-nt", repo_type="dataset", local_dir=TEST_DIR)
    parquets = sorted(glob.glob(f"{TEST_DIR}/**/*.parquet", recursive=True))
print(f"{len(parquets)} parquets test dans {TEST_DIR}")
assert parquets, "aucun parquet test trouvé"

## 5. Inférence (tri par durée, batch dynamique, reprise auto)

In [ ]:
import pandas as pd, time
MAX_SEC_BATCH = 160.0

done = {}
if os.path.exists(PARTIEL):
    dfp = pd.read_csv(PARTIEL)
    done = dict(zip(dfp["id"].astype(str), zip(dfp["language"], dfp["transcription"])))
    print("reprise :", len(done), "déjà transcrits")

rows = [{"id": k, "language": v[0], "transcription": v[1]} for k, v in done.items()]
t0 = time.time()
for pi, pq in enumerate(parquets):
    df = pd.read_parquet(pq)
    items = []
    for _, r in df.iterrows():
        rid = str(r["id"])
        if rid in done: continue
        lang = r.get("language")
        b = r["audio"].get("bytes") if isinstance(r["audio"], dict) else r["audio"]
        try:
            arr = decode_robuste(b)
            items.append((rid, lang, arr, len(arr)/16000.0))
        except Exception:
            rows.append({"id": rid, "language": lang, "transcription": "_"})
    if items:
        items.sort(key=lambda x: x[3])
        i = 0
        while i < len(items):
            j = i + 1
            while j < len(items) and (j - i + 1) * items[j][3] <= MAX_SEC_BATCH:
                j += 1
            batch = items[i:j]
            feats = processor.feature_extractor([x[2] for x in batch], sampling_rate=16000,
                                                return_tensors="pt", padding=True)
            with torch.no_grad():
                logits = model(**{k: v.to(device) for k, v in feats.items()}).logits.cpu().numpy()
            for bi, (rid, lang, _, dur) in enumerate(batch):
                dec = decoders.get(lang) or next(iter(decoders.values()))
                rows.append({"id": rid, "language": lang, "transcription": dec.decode(logits[bi])})
            i = j
    if (pi + 1) % 10 == 0 or pi == len(parquets) - 1:
        pd.DataFrame(rows).to_csv(PARTIEL, index=False)
        print(f"parquet {pi+1}/{len(parquets)} | {len(rows)} transcrits | {(time.time()-t0)/60:.0f} min")
print("✓ inférence terminée :", len(rows))

## 6. Assemblage final + asserts + CSV

In [ ]:
import pandas as pd
sub = pd.DataFrame(rows)
sub["transcription"] = (sub["transcription"].astype(str)
                        .str.replace(r"[\r\n]+", " ", regex=True).str.strip())
sub.loc[sub["transcription"].isin(["", "nan", "None"]), "transcription"] = "_"
sub["id"] = sub["id"].astype(str)
sub = sub.drop_duplicates(subset="id", keep="first")[["id","language","transcription"]]

assert list(sub.columns) == ["id","language","transcription"]
assert len(sub) == 41733, f"attendu 41733, obtenu {len(sub)} — vérifie les parquets/reprise"
assert sub["id"].is_unique
assert sub["language"].notna().all()
assert (sub["transcription"].str.len() > 0).all()

sub.to_csv(FINAL, index=False)
print("✅ soumission écrite ->", FINAL)
print(sub["language"].value_counts())
sub.head(8)

## 7. Après la soumission

- Score attendu ≈ (macro dev-avec-LM du modèle) + ~0.07 (règle de conversion calibrée).
- Pour le modèle suivant (v6-t1, ...) : change UNIQUEMENT la §2 (MODEL_NAME + TAG) et
  relance §1→§6. Si la grille alpha/beta a été refaite et donne un autre optimum,
  mets aussi ALPHA/BETA à jour.
- Le fichier partiel `submission_{TAG}_partiel.csv` peut être supprimé du Drive après
  la soumission réussie.